# Get jobs from the Buildkite API

Ensure the `BUILDKITE_PAT` environmental variable is set before running this notebook.
It can be stored in the `.env` file at the top of the repo, which is kept out of
version control.

In [1]:
import os

if os.path.basename(os.getcwd()) == "notebooks":
    print("Moving to repo root")
    os.chdir("..")

Moving to repo root


In [2]:
import dotenv

dotenv.load_dotenv()

False

In [3]:
import os

token = os.getenv("BUILDKITE_PAT")

if not token:
    raise RuntimeError("No Buildkite token found")

In [4]:
import requests

base_url = "https://api.buildkite.com/v2"

h = {"Authorization": f"Bearer {token}"}

In [5]:
def get_jobs(finished_from: str) -> list[dict]:
    """Get all flagship AMIP jobs from builds finished since ``finished_from``.

    Follows every page of builds, since Buildkite returns 30 at a time by
    default and a longer outage would otherwise skip builds for good.
    """
    jobs = []
    page = 1
    while True:
        resp = requests.get(
            base_url
            + "/organizations/clima/pipelines/climacoupler-coarse-nightly-amip/builds",
            headers=h,
            params={
                "finished_from": finished_from,
                "branch": "main",
                "per_page": 100,
                "page": page,
            },
        )
        resp.raise_for_status()
        builds = resp.json()
        if not builds:
            break
        for b in builds:
            for j in b["jobs"]:
                name = j.get("name") or ""
                if "flagship amip" in name.lower() and "16 helem" in name.lower():
                    job = {
                        "build_id": b.get("id"),
                        "build_state": b.get("state"),
                        "build_created_at": b.get("created_at"),
                        "build_finished_at": b.get("finished_at"),
                        "commit": b.get("commit"),
                        "branch": b.get("branch"),
                        "name": name,
                        "command": j.get("command"),
                        "state": j.get("state"),
                        "created_at": j.get("created_at"),
                        "finished_at": j.get("finished_at"),
                        "id": j.get("id"),
                    }
                    raw_log_url = j.get("raw_log_url")
                    if raw_log_url:
                        log_resp = requests.get(raw_log_url, headers=h)
                        if log_resp.ok:
                            job["raw_log_txt"] = log_resp.text
                        else:
                            # Don't store an error page as if it were the log
                            job["raw_log_status"] = log_resp.status_code
                            print(
                                f"WARN: log for job {j.get('id')}: "
                                f"HTTP {log_resp.status_code}"
                            )
                    jobs.append(job)
        page += 1
    return jobs

In [6]:
from datetime import datetime, timezone
import json

# Let's see what jobs we've already fetched and pick up from there
bk_dir = "buildkite"
min_finished_from = "2026-01-14T00:00:00+00:00"  # First ever successful build

if os.path.basename(os.getcwd()) == "notebooks":
    print("Moving to repo root")
    os.chdir("..")

os.makedirs(bk_dir, exist_ok=True)

# Resume from the newest build finish time already stored. Read it from the
# files rather than their names, which changed to include the job ID.
latest_build_finished = datetime.fromisoformat(min_finished_from)
path_by_job_id = {}
for fname in os.listdir(bk_dir):
    if fname.endswith(".json"):
        with open(os.path.join(bk_dir, fname)) as f:
            stored = json.load(f)
        path_by_job_id[stored.get("id")] = os.path.join(bk_dir, fname)
        finished = stored.get("build_finished_at")
        if finished:
            latest_build_finished = max(
                latest_build_finished,
                datetime.fromisoformat(finished.replace("Z", "+00:00")),
            )

print(f"Fetching jobs from builds finished at {latest_build_finished}")

jobs = get_jobs(
    finished_from=latest_build_finished.astimezone(timezone.utc).isoformat()
)

print(f"Fetched {len(jobs)} jobs")

# Name each file by its build's finish time plus the job ID, so several
# matching jobs in one build don't overwrite each other
for job in jobs:
    finished = datetime.fromisoformat(
        job["build_finished_at"].replace("Z", "+00:00")
    )
    fpath = os.path.join(bk_dir, f"{finished.timestamp()}-{job['id']}.json")
    # A job re-fetched from the newest build may be stored under the old name
    old = path_by_job_id.get(job["id"])
    if old and old != fpath:
        os.remove(old)
    with open(fpath, "w") as f:
        json.dump(job, f, indent=2)

Fetching jobs from builds finished at 2026-11-11 12:26:08.441000+00:00


Fetched 2 jobs
